# Data Exploration - `edgar-corpus`

In [1]:
from pyspark.sql import SparkSession
import os
import sys

# Basic environment variable setup; force 
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("EDGAR_Exploration") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

In [2]:
import datasets

# initial test on small set of data, per example on https://huggingface.co/datasets/eloukas/edgar-corpus
raw_dataset = datasets.load_dataset("eloukas/edgar-corpus", "year_2020", split='train', trust_remote_code=True)
print(raw_dataset.description)
print(raw_dataset.column_names)

c:\Users\Josh\Documents\git-repos\edgar-llm\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



The dataset contains annual filings (10K) of all publicly traded firms from 1993-2020. The table data is stripped but all text is retained.
This dataset allows easy access to the EDGAR-CORPUS dataset based on the paper EDGAR-CORPUS: Billions of Tokens Make The World Go Round (See References in README.md for details).

['filename', 'cik', 'year', 'section_1', 'section_1A', 'section_1B', 'section_2', 'section_3', 'section_4', 'section_5', 'section_6', 'section_7', 'section_7A', 'section_8', 'section_9', 'section_9A', 'section_9B', 'section_10', 'section_11', 'section_12', 'section_13', 'section_14', 'section_15']


In [3]:
# General understanding of shape of dataset
df_spark = spark.createDataFrame(raw_dataset)
df_spark.head(5)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.readRDDFromFile.
: java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.api.java.JavaRDD$.readRDDFromInputStream(JavaRDD.scala:252)
	at org.apache.spark.api.java.JavaRDD$.readRDDFromFile(JavaRDD.scala:239)
	at org.apache.spark.api.python.PythonRDD$.readRDDFromFile(PythonRDD.scala:299)
	at org.apache.spark.api.python.PythonRDD.readRDDFromFile(PythonRDD.scala)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:

In [ ]:
# AIG CIK = 5272 per sec.gov
# Filter from HF dataset rather than spark.
aig_hf = raw_dataset.filter(lambda x: x['cik'] == '5272')
aig = spark.createDataFrame(aig_hf)
aig.show()

Filter: 100%|██████████| 5480/5480 [00:02<00:00, 2109.72 examples/s]


+----+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+
| cik|     filename|           section_1|          section_10|          section_11|          section_12|          section_13|          section_14|          section_15|          section_1A|          section_1B|           section_2|           section_3|           section_4|           section_5|           section_6|           section_7|          section_7A|           section_8|           section_9|          section_9A|          section_9B|year|
+----+-------------+--------------------+--------------------+--------------------+--------------------+----

Py4JJavaError: An error occurred while calling o509.saveAsTextFile.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSyste

In [ ]:
# Quick print out first 350 chars to get a sense of what is in each section
first_row = aig.first().asDict()

for i in first_row.keys():
    print(i+'\n'+str(len(first_row[i]))+'\n'+first_row[i][0:349]+'\n\n')

cik
4
5272


filename
13
5272_2020.htm


section_1
42579
ITEM 1 | Business
DOL Fiduciary Rule
In June 2020, the DOL issued final guidance consistent with the June 2018 decision of the Fifth Circuit Court of Appeals which vacated the DOL’s Fiduciary Rule that redefined who would be considered a “fiduciary” for purposes of transactions with ERISA qualified plans, related plan participants and Individual R


section_10
378
ITEM 10 | Directors, Executive Officers and Corporate Governance
All information required by Items 10, 11, 12, 13 and 14 of this Form 10-K is incorporated by reference from the definitive proxy statement for AIG’s 2021 Annual Meeting of Shareholders, which will be filed with the SEC not later than 120 days after the close of the fiscal year pursua


section_11
60
ITEM 11 | Executive Compensation
See Item 10 herein.
ITEM 12


section_12
132
ITEM 12 | Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters
See Item 10 herein.
ITEM 13

In [ ]:
from pyspark.sql.functions import col, split, explode, monotonically_increasing_id, instr

def retrieve_context_chunks(df_spark, target_section, keyword):
    """
    Splits a massive section column into paragraphs and retrieves 
    the specific text chunk containing the keyword.
    """
    # 1. Break the massive block string down by double-newlines (paragraphs)
    df_split = df_spark.withColumn("paragraphs", split(col(target_section), "\n"))
    
    # 2. Explode the array column so every paragraph becomes an independent Row
    df_exploded = df_split.withColumn("chunk_text", explode(col("paragraphs")))
    
    # 3. Add a unique row identifier to represent "virtual pages/blocks"
    df_indexed = df_exploded.withColumn("chunk_id", monotonically_increasing_id())
    
    # 4. Filter strictly down to chunks matching your retrieval query
    df_matches = df_indexed.filter(instr(col("chunk_text"), keyword) > 0)
    
    return df_matches.select("chunk_id", "chunk_text")


retrieve_context_chunks(aig, 'section_8', 'total revenue').show()

+-----------+--------------------+
|   chunk_id|          chunk_text|
+-----------+--------------------+
|25769804108|The accounting po...|
|25769804388|The following tab...|
+-----------+--------------------+

